# Notebook 05 — V5: ¿Quién llega al sistema? (barras apiladas)

**Visualización:** Barras apiladas al 100% en Flourish  
**Pregunta:** Entre las personas con mala salud mental, ¿quién accede al sistema sanitario y quién no? ¿Depende del nivel de ingresos?  
**Output:** `data/processed/v5_barras_acceso.csv`

### Notas de codificación en ESS R11
- `fltdpr` / `fltlnl`: escala **1-4** (1=nunca, 4=siempre), NO 0-10 — umbral mala salud: **≥ 3**
- `hinctnta`: los deciles **6-9 fueron eliminados** por el limpiador genérico de missings; se re-leen del CSV crudo
- `hltprhc`: 1 = ha necesitado atención de salud mental; `hltphhc`: 1 = la recibió
- `dshltms`: 1 = razón económica es la principal barrera percibida para acceder

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../')
OUT  = ROOT / 'data' / 'processed'
RAW11 = ROOT / 'ESS11' / 'ESS11e04_1.csv'

print('Setup OK')

Setup OK


## 1. Cargar ESS R11 limpio y corregir `hinctnta`

In [2]:
ess11 = pd.read_csv(OUT / 'ESS11_clean.csv', low_memory=False)
print(f'ESS R11 limpio: {len(ess11):,} filas')

# hinctnta en el CSV limpio solo tiene deciles 1-5 y 10 (6-9 eliminados por error)
print(f'hinctnta en CSV limpio: {sorted(ess11["hinctnta"].dropna().unique())} → {ess11["hinctnta"].notna().sum():,} válidos')

# Re-leer hinctnta del CSV crudo, aplicando SOLO los missing codes correctos (77, 88, 99)
raw_inc = pd.read_csv(RAW11, usecols=['hinctnta'], low_memory=False)
raw_inc['hinctnta_fix'] = pd.to_numeric(raw_inc['hinctnta'], errors='coerce')
raw_inc.loc[raw_inc['hinctnta_fix'].isin([77, 88, 99]), 'hinctnta_fix'] = np.nan

print(f'\nhinctnta corregido (deciles 1-10): {sorted(raw_inc["hinctnta_fix"].dropna().unique())} → {raw_inc["hinctnta_fix"].notna().sum():,} válidos')

ess11['hinctnta_dec'] = raw_inc['hinctnta_fix'].values
print(f'Cobertura hinctnta_dec: {ess11["hinctnta_dec"].notna().sum():,} ({ess11["hinctnta_dec"].notna().mean()*100:.1f}%)')

ESS R11 limpio: 50,116 filas
hinctnta en CSV limpio: [1.0, 2.0, 3.0, 4.0, 5.0, 10.0] → 23,554 válidos



hinctnta corregido (deciles 1-10): [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0] → 39,688 válidos
Cobertura hinctnta_dec: 39,688 (79.2%)


## 2. Identificar población con mala salud mental

In [3]:
# fltdpr y fltlnl en ESS R11: escala 1-4 (NO 0-10)
# 1=nunca/casi nunca  2=a veces  3=la mayor parte  4=siempre/casi siempre
# health: 1=muy buena ... 5=muy mala
# Umbral "mala salud mental": frecuencia alta de síntomas O salud autopercibida mala
mala_sm = (
    (ess11['fltdpr'] >= 3) |   # deprimido la mayor parte o siempre
    (ess11['fltlnl'] >= 3) |   # solo la mayor parte o siempre
    (ess11['health'] >= 4)     # salud mala o muy mala
)

print(f'Con mala salud mental (criterio ≥3 síntomas o salud≥4): {mala_sm.sum():,} ({mala_sm.mean()*100:.1f}%)')
print()
print('Desglose por componente:')
print(f'  fltdpr >= 3: {(ess11["fltdpr"]>=3).sum():,}')
print(f'  fltlnl >= 3: {(ess11["fltlnl"]>=3).sum():,}')
print(f'  health >= 4: {(ess11["health"]>=4).sum():,}')
print()
pop_mala = ess11[mala_sm].copy()
print(f'Hombres: {(pop_mala["gndr"]==1).sum():,}  |  Mujeres: {(pop_mala["gndr"]==2).sum():,}')

Con mala salud mental (criterio ≥3 síntomas o salud≥4): 8,886 (17.7%)

Desglose por componente:
  fltdpr >= 3: 3,768
  fltlnl >= 3: 4,561
  health >= 4: 3,908

Hombres: 3,554  |  Mujeres: 5,332


## 3. Clasificar en 4 categorías de acceso

In [4]:
# hltprhc=1: necesitó atención de salud mental en los últimos 12 meses
# hltphhc=1: la recibió cuando la necesitó
# dshltms=1: la barrera económica es la razón principal para no acceder

def clasificar_acceso(row):
    if row['hltprhc'] == 1 and row['hltphhc'] == 1:
        return 'accede'                    # buscó y recibió ayuda profesional
    elif row['hltprhc'] == 1 and row['hltphhc'] == 0:
        return 'busco_no_recibio'          # buscó pero no consiguió atención
    elif row['dshltms'] == 1:
        return 'no_accede_economico'       # no buscó, barrera económica
    else:
        return 'no_accede_otro'            # no buscó, otra razón

pop_mala['categoria_acceso'] = pop_mala.apply(clasificar_acceso, axis=1)

print('Distribución de categorías de acceso (sobre mala salud mental):')
print(pop_mala['categoria_acceso'].value_counts())
print()
print('Proporciones:')
print((pop_mala['categoria_acceso'].value_counts() / len(pop_mala) * 100).round(1))

Distribución de categorías de acceso (sobre mala salud mental):
categoria_acceso
no_accede_otro         3332
no_accede_economico    3262
accede                 1371
busco_no_recibio        921
Name: count, dtype: int64

Proporciones:
categoria_acceso
no_accede_otro         37.5
no_accede_economico    36.7
accede                 15.4
busco_no_recibio       10.4
Name: count, dtype: float64


## 4. Filtrar con hinctnta válido y agrupar

In [5]:
# Filtrar solo los que tienen decil de ingresos válido
pop_v5 = pop_mala.dropna(subset=['hinctnta_dec']).copy()
pop_v5['decil'] = pop_v5['hinctnta_dec'].astype(int)
pop_v5['genero'] = pop_v5['gndr'].map({1: 'Hombres', 2: 'Mujeres'})

print(f'Con mala salud mental Y hinctnta válido: {len(pop_v5):,} ({len(pop_v5)/mala_sm.sum()*100:.1f}% del grupo)')
print(f'  Hombres: {(pop_v5["genero"]=="Hombres").sum():,}  |  Mujeres: {(pop_v5["genero"]=="Mujeres").sum():,}')
print()
print('N por decil:')
print(pop_v5.groupby('decil').size().to_dict())

Con mala salud mental Y hinctnta válido: 7,232 (81.4% del grupo)
  Hombres: 2,891  |  Mujeres: 4,341

N por decil:
{1: 1474, 2: 1196, 3: 1039, 4: 837, 5: 692, 6: 603, 7: 529, 8: 373, 9: 247, 10: 242}


In [6]:
# Calcular proporciones por genero × decil
tabla = (
    pop_v5.groupby(['genero', 'decil', 'categoria_acceso'])
    .size()
    .reset_index(name='n')
)

# Total por genero × decil
totales = pop_v5.groupby(['genero', 'decil']).size().reset_index(name='n_total')
tabla = tabla.merge(totales, on=['genero', 'decil'])
tabla['pct'] = (tabla['n'] / tabla['n_total'] * 100).round(2)

# Pivotar a formato wide para Flourish
v5 = tabla.pivot_table(
    index=['genero', 'decil', 'n_total'],
    columns='categoria_acceso',
    values='pct',
    fill_value=0
).reset_index()
v5.columns.name = None

# Asegurar que todas las columnas de categoría existen
for cat in ['accede', 'busco_no_recibio', 'no_accede_economico', 'no_accede_otro']:
    if cat not in v5.columns:
        v5[cat] = 0.0

# Renombrar para Flourish
v5 = v5.rename(columns={
    'accede':              'pct_accede',
    'busco_no_recibio':    'pct_busco_no_recibio',
    'no_accede_economico': 'pct_no_accede_eco',
    'no_accede_otro':      'pct_no_accede_otro',
    'n_total':             'n'
})

v5 = v5.sort_values(['genero', 'decil']).reset_index(drop=True)

print(f'Filas del CSV: {len(v5)} (2 géneros × 10 deciles = 20)')
print(v5.to_string(index=False))

Filas del CSV: 20 (2 géneros × 10 deciles = 20)
 genero  decil   n  pct_accede  pct_busco_no_recibio  pct_no_accede_eco  pct_no_accede_otro
Hombres      1 489       20.45                  9.00              26.58               43.97
Hombres      2 419       23.15                  9.55              32.22               35.08
Hombres      3 409       17.36                 14.91              32.52               35.21
Hombres      4 357       17.65                 11.20              35.01               36.13
Hombres      5 286       13.29                 11.54              34.97               40.21
Hombres      6 279       14.70                 10.39              42.29               32.62
Hombres      7 258       13.57                  8.91              29.46               48.06
Hombres      8 154       12.99                  5.84              41.56               39.61
Hombres      9 113        5.31                  7.08              40.71               46.90
Hombres     10 127       10.24  

## 5. Exploración: gradiente de ingresos

In [7]:
print('=== Gradiente de acceso por decil (ambos géneros combinados) ===')
grad = (
    pop_v5.groupby(['decil', 'categoria_acceso'])
    .size().reset_index(name='n')
)
tot_dec = pop_v5.groupby('decil').size().reset_index(name='n_total')
grad = grad.merge(tot_dec, on='decil')
grad['pct'] = (grad['n'] / grad['n_total'] * 100).round(1)

pivot_grad = grad.pivot(index='decil', columns='categoria_acceso', values='pct').fillna(0)
cols_order = [c for c in ['accede', 'busco_no_recibio', 'no_accede_economico', 'no_accede_otro'] if c in pivot_grad.columns]
print(pivot_grad[cols_order].to_string())
print()

# Decil 1 vs decil 10
print('Decil 1 (más pobre) vs Decil 10 (más rico) — % que ACCEDE:')
for d in [1, 10]:
    pct = pivot_grad.loc[d, 'accede'] if d in pivot_grad.index else 0
    n = tot_dec[tot_dec['decil']==d]['n_total'].values[0]
    print(f'  Decil {d}: {pct:.1f}%  (n={n})')

=== Gradiente de acceso por decil (ambos géneros combinados) ===
categoria_acceso  accede  busco_no_recibio  no_accede_economico  no_accede_otro
decil                                                                          
1                   21.8              12.1                 30.3            35.7
2                   21.7              12.0                 33.4            32.9
3                   17.3              15.2                 34.6            32.9
4                   15.8              12.5                 38.4            33.3
5                   12.3               9.4                 40.8            37.6
6                   13.6               7.8                 43.6            35.0
7                   10.6               9.3                 37.2            42.9
8                   10.5               6.2                 45.0            38.3
9                    5.7               7.3                 47.8            39.3
10                   6.6               4.1             

In [8]:
print('=== Por género: % que accede según decil ===')
for genero in ['Hombres', 'Mujeres']:
    sub = v5[v5['genero']==genero][['decil','pct_accede','pct_no_accede_eco','n']]
    print(f'\n{genero}:')
    print(sub.to_string(index=False))

=== Por género: % que accede según decil ===

Hombres:
 decil  pct_accede  pct_no_accede_eco   n
     1       20.45              26.58 489
     2       23.15              32.22 419
     3       17.36              32.52 409
     4       17.65              35.01 357
     5       13.29              34.97 286
     6       14.70              42.29 279
     7       13.57              29.46 258
     8       12.99              41.56 154
     9        5.31              40.71 113
    10       10.24              37.01 127

Mujeres:
 decil  pct_accede  pct_no_accede_eco   n
     1       22.54              32.18 985
     2       20.85              34.11 777
     3       17.30              35.87 630
     4       14.37              40.83 480
     5       11.58              44.83 406
     6       12.65              44.75 324
     7        7.75              44.65 271
     8        8.68              47.49 219
     9        5.97              53.73 134
    10        2.61              62.61 115


## 6. Verificar y guardar

In [9]:
def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)} | Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos:\n{nulos_con}')
    else:
        print('Sin valores nulos')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    # Las proporciones deben sumar ~100 por fila
    pct_cols = [c for c in df.columns if c.startswith('pct_')]
    sumas = df[pct_cols].sum(axis=1)
    assert (sumas - 100).abs().max() < 1, f'Proporciones no suman 100: {sumas.describe()}'
    print(f'Suma proporciones: min={sumas.min():.1f}, max={sumas.max():.1f} ✓')
    print(f'Muestra:\n{df.head(4).to_string()}')
    print('✓ OK')

verificar_csv_flourish(v5, 'v5_barras_acceso')

out_path = OUT / 'v5_barras_acceso.csv'
v5.to_csv(out_path, index=False)
print(f'\nGuardado: {out_path}')


=== v5_barras_acceso ===
Filas: 20 | Columnas: ['genero', 'decil', 'n', 'pct_accede', 'pct_busco_no_recibio', 'pct_no_accede_eco', 'pct_no_accede_otro']
Sin valores nulos
Suma proporciones: min=100.0, max=100.0 ✓
Muestra:
    genero  decil    n  pct_accede  pct_busco_no_recibio  pct_no_accede_eco  pct_no_accede_otro
0  Hombres      1  489       20.45                  9.00              26.58               43.97
1  Hombres      2  419       23.15                  9.55              32.22               35.08
2  Hombres      3  409       17.36                 14.91              32.52               35.21
3  Hombres      4  357       17.65                 11.20              35.01               36.13
✓ OK

Guardado: ../data/processed/v5_barras_acceso.csv


## 7. Resumen

In [10]:
print('=' * 60)
print('RESUMEN V5 — BARRAS ACCESO AL SISTEMA')
print('=' * 60)
print()
print(f'Base: personas con mala salud mental: {mala_sm.sum():,}')
print(f'Con hinctnta válido (para visualización): {len(pop_v5):,}')
print()
print('Distribución global de categorías:')
cats = pop_mala['categoria_acceso'].value_counts()
for cat, n in cats.items():
    print(f'  {cat}: {n:,} ({n/len(pop_mala)*100:.1f}%)')
print()

# Gradiente decil 1 vs 10
d1  = v5[v5['decil']==1][['genero','pct_accede','pct_no_accede_eco','n']]
d10 = v5[v5['decil']==10][['genero','pct_accede','pct_no_accede_eco','n']]
print('Decil 1 (más pobre):')
print(d1.to_string(index=False))
print('\nDecil 10 (más rico):')
print(d10.to_string(index=False))
print()
print('Configuración Flourish:')
print('  Tipo: Bar chart apilado al 100%')
print('  Eje X: decil (1-10)')
print('  Segmentos: pct_accede / pct_busco_no_recibio / pct_no_accede_eco / pct_no_accede_otro')
print('  Panel/filtro: genero (dos gráficos en paralelo)')
print('  Paleta: verde(accede) / amarillo(buscó,no recibió) / naranja(eco) / gris(otro)')
print('  Anotaciones: destacar decil 1 y 10 para mostrar la brecha')

RESUMEN V5 — BARRAS ACCESO AL SISTEMA

Base: personas con mala salud mental: 8,886
Con hinctnta válido (para visualización): 7,232

Distribución global de categorías:
  no_accede_otro: 3,332 (37.5%)
  no_accede_economico: 3,262 (36.7%)
  accede: 1,371 (15.4%)
  busco_no_recibio: 921 (10.4%)

Decil 1 (más pobre):
 genero  pct_accede  pct_no_accede_eco   n
Hombres       20.45              26.58 489
Mujeres       22.54              32.18 985

Decil 10 (más rico):
 genero  pct_accede  pct_no_accede_eco   n
Hombres       10.24              37.01 127
Mujeres        2.61              62.61 115

Configuración Flourish:
  Tipo: Bar chart apilado al 100%
  Eje X: decil (1-10)
  Segmentos: pct_accede / pct_busco_no_recibio / pct_no_accede_eco / pct_no_accede_otro
  Panel/filtro: genero (dos gráficos en paralelo)
  Paleta: verde(accede) / amarillo(buscó,no recibió) / naranja(eco) / gris(otro)
  Anotaciones: destacar decil 1 y 10 para mostrar la brecha
